In [1]:
from pynq import Overlay, MMIO
import pynq.lib as lib
from pynq import allocate
import numpy as np
from pynq.lib import DMA

ol = Overlay("DMA_0513_wrapper.bit")

In [69]:
ol.axi_dma_0?

In [83]:
ol.ip_dict.keys()

dict_keys(['axi_dma_0', 'zynq_ultra_ps_e_0'])

In [181]:
dma = ol.axi_dma_0
dma.register_map.MM2S_DMACR

Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0)

In [85]:
type(dma)

pynq.lib.dma.DMA

In [49]:
# dma.read(0x00)      # 讀 MM2S_DMACR
# dma.read(0x04)      # 讀 MM2S_DMASR
# dma.write(0x00, 0x1)  # 啟動 DMA

In [50]:
# 注意這邊會用到MMIO來開DMA，而不是lib的方式
# dma = MMIO(0xA0000000, 0x10000)
# val = dma.read(0x00)
# print("DMA MM2S_DMACR =", hex(val))

DMA MM2S_DMACR = 0x10003


In [143]:
# 分配緩存區
data_size = 100
TX_buffer = allocate(shape = (data_size,), dtype = np.uint32)
RX_buffer = allocate(shape = (data_size,), dtype = np.uint32)
print("TX Buf Addr:", hex(TX_buffer.physical_address))
print("RX Buf Addr:", hex(RX_buffer.physical_address))
print("---")
print("DMA Source address     :", hex(dma.register_map.MM2S_SA.Source_Address))
print("DMA Destination address:", hex(dma.register_map.S2MM_DA.Destination_Address))

TX Buf Addr: 0x95c8000
RX Buf Addr: 0x98d8000
---
DMA Source address     : 0x0
DMA Destination address: 0x0


In [144]:
# 將資料寫入Buffer
for i in range(100):
    TX_buffer[i] = i

In [145]:
# 從 DDR 中的記憶體區塊到 FIFO 執行 DMA 傳輸
dma.sendchannel.transfer(TX_buffer)
dma.recvchannel.transfer(RX_buffer)

dma.sendchannel.wait()
dma.recvchannel.wait()


In [146]:
for i in range(100):
    print(RX_buffer[i])

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [114]:
print("Arrays are equal: {}".format(np.array_equal(TX_buffer, RX_buffer)))

Arrays are equal: True


In [71]:
dma.register_map

RegisterMap {
  MM2S_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  MM2S_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SGDecErr=0, IOC_Irq=1, Dly_Irq=0, Err_Irq=0, IRQThresholdSts=1, IRQDelaySts=0),
  MM2S_CURDESC = Register(Current_Descriptor_Pointer=704065),
  MM2S_CURDESC_MSB = Register(Current_Descriptor_Pointer=0),
  MM2S_TAILDESC = Register(Tail_Descriptor_Pointer=704065),
  MM2S_TAILDESC_MSB = Register(Tail_Descriptor_Pointer=0),
  MM2S_SA = Register(Source_Address=0),
  MM2S_SA_MSB = Register(Source_Address=0),
  MM2S_LENGTH = Register(Length=0),
  SG_CTL = Register(SG_CACHE=3, SG_USER=0),
  S2MM_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  S2MM_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSl

In [140]:
del TX_buffer, RX_buffer

In [174]:
from pynq import allocate
import numpy as np
import time

class DMALoopbackTest:
    def __init__(self, dma):
        self.dma = dma

    def LPT(self, data_size):
        with allocate(shape=(data_size,), dtype=np.uint32) as in_buffer, \
             allocate(shape=(data_size,), dtype=np.uint32) as out_buffer:

            for i in range(data_size):
                in_buffer[i] = i
            start = time.time()
            self.dma.sendchannel.transfer(in_buffer)
            self.dma.recvchannel.transfer(out_buffer)

            self.dma.sendchannel.wait()
            self.dma.recvchannel.wait()
            end = time.time()
            print('time cost ' + str(round(end - start, 5)) + 's')
            result = out_buffer.copy()
        return result

In [178]:
tester = DMALoopbackTest(dma)
result = tester.LPT(100)
print(result)
# for i in range(100):
#     print(result[i])

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]


In [183]:
# DMA 速度測試
from pynq import allocate
import numpy as np
import time

in_buffer = allocate(shape=(data_size,), dtype=np.uint32)
out_buffer = allocate(shape=(data_size,), dtype=np.uint32)

# 25MB * 4 (width)
# 且dma每個clock搬32-bit (Data Width)
data_size = 6553600

for i in range(data_size):
    in_buffer[i] = i
start = time.time()
dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)

dma.sendchannel.wait()
dma.recvchannel.wait()
end = time.time()

print("Arrays are equal: {}".format(np.array_equal(in_buffer, out_buffer)))
print('time cost ' + str(round(end - start, 5)) + 's')
# 25MB
length = 26214400
cost = round(end - start, 5)
speed = round(length/(cost*1048576),5) #換算MB
print('transfer speed: ' + str(speed) + 'MB/s')

Arrays are equal: True
time cost 0.266s
transfer speed: 93.98496MB/s
